In [1]:
import pandas as pd  # importa pandas, la libreria per lavorare con dati in tabelle
df = pd.read_csv('../../top_rated_wines.csv')  # legge il file CSV dei vini e lo carica in una tabella (DataFrame)
df = df[df['variety'].notna()]  # tiene solo le righe con la colonna 'variety' compilata (i valori vuoti/NaN darebbero errore dopo)
data = df.to_dict('records')  # converte la tabella in una lista di dizionari: un dizionario per ogni vino
df  # mostra la tabella a schermo

,name,region,variety,rating,notes
0,3 Rings Reserve Shiraz 2004,"Barossa Valley, Barossa, South Australia, Aust...",Red Wine,96.0,Vintage Comments : Classic Barossa vintage con...
1,Abreu Vineyards Cappella 2007,"Napa Valley, California",Red Wine,96.0,Cappella is a proprietary blend of two clones ...
2,Abreu Vineyards Cappella 2010,"Napa Valley, California",Red Wine,98.0,Cappella is one of the oldest vineyard sites i...
3,Abreu Vineyards Howell Mountain 2008,"Howell Mountain, Napa Valley, California",Red Wine,96.0,When David purchased this Howell Mountain prop...
4,Abreu Vineyards Howell Mountain 2009,"Howell Mountain, Napa Valley, California",Red Wine,98.0,"As a set of wines, it is hard to surpass the f..."
...,...,...,...,...,...
1360,Lewis Cellars Alec's Blend Red 2002,"Napa Valley, California",Red Wine,96.0,Number 12 on
1361,Lewis Cellars Cabernet Sauvignon 2002,"Napa Valley, California",Red Wine,96.0,Showcasing the unique personalities of small h...
1362,Lewis Cellars Cuvee L Cabernet Sauvignon 2015,"Napa Valley, California",Red Wine,96.0,"Straight from James Fenimore Cooper’s novel, L..."
1363,Lewis Cellars Reserve Cabernet Sauvignon 2010,"Napa Valley, California",Red Wine,96.0,


In [2]:
from qdrant_client import models, QdrantClient  # importa Qdrant: il database che salva e cerca vettori
from sentence_transformers import SentenceTransformer  # importa la libreria che trasforma il testo in vettori (embedding)

C:\Users\lucam\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
encoder = SentenceTransformer('all-MiniLM-L6-v2')  # carica un modello già addestrato che trasforma una frase in una lista di 384 numeri (il suo "significato")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5002.41it/s]


In [4]:
# crea il client del database vettoriale
qdrant = QdrantClient(":memory:")  # avvia Qdrant in memoria: niente server, i dati spariscono quando chiudi il notebook

In [5]:
# crea la "collection" (come una tabella) dove salvare i vini
qdrant.recreate_collection(  # crea la collection; se esiste già la cancella e la rifà da zero
    collection_name="top_wines",  # nome della collection
    vectors_config=models.VectorParams(  # dice a Qdrant com'è fatto ogni vettore
        size=encoder.get_sentence_embedding_dimension(),  # lunghezza del vettore = quella prodotta dal modello (384)
        distance=models.Distance.COSINE  # misura la somiglianza tra vettori con la distanza coseno
    )
)

C:\Users\lucam\AppData\Local\Temp\ipykernel_28668\2622090487.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  size=encoder.get_sentence_embedding_dimension(),  # lunghezza del vettore = quella prodotta dal modello (384)
C:\Users\lucam\AppData\Local\Temp\ipykernel_28668\2622090487.py:2: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(  # crea la collection; se esiste già la cancella e la rifà da zero


True

In [6]:
# trasforma in vettori e carica nel database
qdrant.upload_points(  # carica tanti "punti" nella collection in una volta sola
    collection_name="top_wines",  # la collection in cui caricarli
    points=[  # lista dei punti, uno per vino
        models.PointStruct(  # un punto = id + vettore + dati allegati
            id=idx,  # identificativo numerico del vino (0, 1, 2, ...)
            vector=encoder.encode(doc["notes"]).tolist(),  # trasforma le note di degustazione in vettore e lo converte in lista Python
            payload=doc  # salva insieme al vettore tutti i dati del vino (nome, regione, voto, ...)
        ) for idx, doc in enumerate(data)  # ripete per ogni vino in 'data', con idx = posizione nella lista
    ]
)

In [13]:
# ora cerchiamo dei vini

response = qdrant.query_points(  # cerca nella collection i vettori più simili a quello della domanda
    collection_name="top_wines",  # in quale collection cercare
    query=encoder.encode("Wines frome Tuscany").tolist(),  # trasforma la domanda in vettore con lo stesso modello
    limit=3  # restituisce solo i 3 risultati più simili
)
for hit in response.points:  # per ogni risultato trovato...
  print(hit.payload, "score:", hit.score)  # ...stampa i dati del vino e il punteggio di somiglianza (più vicino a 1 = più simile)

{'name': 'Bodegas Vega Sicilia Unico Tinto 1998', 'region': 'Ribera del Duero, Spain', 'variety': 'Red Wine', 'rating': 98.0, 'notes': 'It comes from the oldest vines on the property and in its composition, in addition to tempranillo are cabernet sauvignon and merlot.  Although it is old, it maintains as a lively red wine.  It has an intense ripe cherry color.  Hints of toasted wood notes. '} score: 0.6645022499323916
{'name': 'Fontodi Flaccianello della Pieve 2015', 'region': 'Tuscany, Italy', 'variety': 'Red Wine', 'rating': 96.0, 'notes': 'Flaccianello combines all the wild and old-world characteristics of the Sangiovese grape with the modern vinification techniques of what may be the top winemaking house in Tuscany today. The vineyards from which this wine takes its name continues to produce a Sangiovese of superior quality year on year. Made with 100% Sangiovese.'} score: 0.6353238616513862
{'name': 'Glaetzer Anaperenna Shiraz/Cabernet Sauvignon 2006', 'region': 'Barossa Valley, B